# Sentiment Analysis using Bidirectional LSTM with Word2Vec

## CSE 4221 — Natural Language Processing Assignment

**Model:** Bidirectional LSTM  
**Embeddings:** Word2Vec (300-dimensional, trained on corpus)  
**Dataset:** IMDB Movie Review Dataset  
**Task:** Binary Sentiment Classification (Positive / Negative)

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported successfully.')

---
## 2. Load the IMDB Dataset

In [ ]:
df = pd.read_csv('IMDB-Dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nSentiment Distribution:\n{df["sentiment"].value_counts()}')

# Remove duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed {duplicates} duplicates. New shape: {df.shape}')

df.head()

---
## 3. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Clean and preprocess a single review."""
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

print('Preprocessing reviews...')
df['cleaned_review'] = df['review'].apply(preprocess_text)
print('Preprocessing complete.')

# Encode labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(f'\nSample cleaned review:\n{df["cleaned_review"].iloc[0][:150]}')

---
## 4. Train-Test Split (80:20)

In [ ]:
X = df['cleaned_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)}')
print(f'Test set:     {len(X_test)}')

---
## 5. Tokenization and Sequence Padding

We tokenize the text and convert each review into a fixed-length sequence of integer indices.

In [ ]:
# Parameters
VOCAB_SIZE = 20000
MAX_LEN = 200    # Maximum sequence length
EMBED_DIM = 300  # Word2Vec embedding dimension

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

word_index = tokenizer.word_index
print(f'Vocabulary size: {len(word_index)}')
print(f'Training sequences shape: {X_train_pad.shape}')
print(f'Test sequences shape:     {X_test_pad.shape}')

---
## 6. Train Word2Vec Embeddings

We train a **Word2Vec** model on the training corpus to learn 300-dimensional word embeddings. These embeddings capture semantic relationships between words based on their co-occurrence patterns.

In [ ]:
# Prepare tokenized sentences for Word2Vec
train_sentences = [text.split() for text in X_train]

# Train Word2Vec model
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    sg=0  # CBOW (Continuous Bag of Words)
)

print(f'Word2Vec vocabulary size: {len(w2v_model.wv)}')
print(f'Embedding dimension: {w2v_model.wv.vector_size}')

In [ ]:
# Build embedding matrix
vocab_size = min(VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

found = 0
not_found = 0
for word, idx in word_index.items():
    if idx >= vocab_size:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        found += 1
    else:
        not_found += 1

print(f'Embedding matrix shape: {embedding_matrix.shape}')
print(f'Words found in Word2Vec: {found}')
print(f'Words NOT found (zero vectors): {not_found}')
print(f'Coverage: {found/(found+not_found)*100:.1f}%')

---
## 7. Build Bidirectional LSTM Model

The Bi-LSTM processes the input sequence in both forward and backward directions, allowing it to capture context from both sides of each word.

In [ ]:
model = Sequential([
    # Pre-trained Word2Vec embeddings (non-trainable initially)
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    SpatialDropout1D(0.3),
    
    # Bidirectional LSTM layer
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    
    # Dense classification layers
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## 8. Train the Model

In [ ]:
# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Convert labels to numpy arrays
y_train_arr = y_train.values
y_test_arr = y_test.values

# Train the model
history = model.fit(
    X_train_pad, y_train_arr,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Prediction and Evaluation

In [ ]:
# Predict on test set
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

accuracy  = accuracy_score(y_test_arr, y_pred)
precision = precision_score(y_test_arr, y_pred)
recall    = recall_score(y_test_arr, y_pred)
f1        = f1_score(y_test_arr, y_pred)

print('=' * 55)
print('  Bi-LSTM + Word2Vec — RESULTS')
print('=' * 55)
print(f'  Accuracy:  {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1-Score:  {f1:.4f}')
print('=' * 55)

In [ ]:
# Classification Report
print('\nClassification Report:')
print(classification_report(y_test_arr, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test_arr, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(cmap='Greens', ax=ax, values_format='d')
ax.set_title('Confusion Matrix — Bi-LSTM + Word2Vec', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Analysis and Discussion

### Observations
- **Word2Vec embeddings** capture semantic relationships between words, providing dense 300-dim representations.
- **Bidirectional LSTM** processes sequences in both directions, capturing context from past and future words.
- The stacked Bi-LSTM architecture (128 + 64 units) learns hierarchical representations of the text.

### Advantages
- Dense embeddings encode semantic similarity (e.g., "good" and "great" are nearby).
- Bi-LSTM captures word order and long-range dependencies — critical for sentiment.
- The model can learn complex non-linear patterns in the data.

### Limitations
- Word2Vec is trained only on the IMDB corpus, which limits vocabulary coverage.
- LSTM training is computationally expensive compared to classical ML methods.
- Requires tuning of many hyperparameters (sequence length, LSTM units, dropout, etc.).

### Conclusion
The Bi-LSTM + Word2Vec approach leverages both semantic embeddings and sequential modeling, and is expected to outperform BoW/TF-IDF baselines by capturing contextual information and word order.